**Bank Marketing 데이터셋**

고객의 인구통계적 정보 및 이전 마케팅 이력 데이터를 바탕으로 "이 사람이 정기예금 상품에 가입할 것인가?" 를 예측하는 것이 목적입니다.

제공된 학습용 데이터(bank_train.csv)를 이용하여 정기예금 상품에 가입 여부를 예측하는 모델을 개발하고, 개발한 모델에 기반하여 평가용 데이터(bank_test.csv)에 적용하여 얻은 정기예금 상품에 가입 여부 예측 확률을 아래 [제출형식]에 따라 csv 파일로 생성하여 제출하시오.
- 예측 결과는 ROC-AUC 평가지표에 따라 평가함
- 성능이 우수한 예측 모델을 구축하기 위해서는 데이터 정제, Feature Engineering, 하이퍼 파라미터(hyper parameter) 최적화, 모델 비교 등이 필요할 수 있음. 다만, 과적합에 유의하여야 함


[[제출 형식]]
- 가. CSV 파일명: result.csv(파일명에 디렉토리/폴더 지정불가
- 나. 예측 성별 칼럼명 : pred
- 다. 제출 칼럼 개수 : pred 칼럼 1개
- 라. 평가용 데이터 개수와 예측 결과 데이터 개수 일치 : 1,221개

[[제공 데이터]]
- 데이터 목록
- bank_train.csv : 학습용 데이터, 3,300개
- bank_test.csv : 평가용 데이터, 1,221개
- 평가용 데이터는 'term_deposit' 칼럼 미제공

In [2]:
# 라이브러리
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# 데이터 확인
path = "https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/"
train = pd.read_csv(path + "bank_train.csv")
test =  pd.read_csv(path + "bank_test.csv")
# print(train.head(3), train.shape, sep="\n") # (3300, 14)
# print(test.head(3), test.shape, sep="\n")   # (1221, 13)

# 데이터 전처리
X = train.drop(columns=['term_deposit'])
Y = train['term_deposit']
X_all = pd.concat([X, test])
# print(X_all.head(3))
cols_obj = X_all.select_dtypes(include='object').columns
# print(cols_obj)
for col in cols_obj:
    X_all[col] = LabelEncoder().fit_transform(X_all[col])
# print(X_all.head(3))
# X_all = pd.get_dummies(X_all, drop_first=True, dtype='int')
# print(X_all.head(3))

# 데이터 분할
X = X_all.iloc[:len(X), :]
X_submission = X_all.iloc[len(X):, :]
# print(X.shape, X_submission.shape) # (3300, 13) (1221, 13)
temp = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=123)
x_train, x_test, y_train, y_test = temp
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape) # (2640, 13) (660, 13) (2640,) (660,)

# 파이프라인 모델사전 생성
models = {
    "Logistic": Pipeline([
        ('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, tol=0.05, random_state=123))
    ]),
    "DecisionTree": Pipeline([
        ('model', DecisionTreeClassifier(max_depth=3, random_state=123))
    ]),
    "RandomForet": Pipeline([
        ('model', RandomForestClassifier(max_depth=3, random_state=123))
    ]),
    "AdaBoost": Pipeline([
        ('model', AdaBoostClassifier(n_estimators=500, random_state=123))
    ]),
    "GradientBoosting": Pipeline([
        ('model', GradientBoostingClassifier(random_state=123))
    ])
}

# 성능평가 함수
def get_scores(model, x_train, x_test, y_train, y_test):
    model.fit(x_train, y_train)
    y_proba1 = model.predict_proba(x_train)[:, 1]
    y_proba2 = model.predict_proba(x_test)[:, 1]
    AUC_train = roc_auc_score(y_train, y_proba1)
    AUC_test = roc_auc_score(y_test, y_proba2)
    return model, AUC_train, AUC_test

# 모델적합 성능평가
results = []
for name, model in models.items():
    model, AUC_train, AUC_test = get_scores(model, x_train, x_test, y_train, y_test)
    results.append({
        "Model": name, "AUC_train": AUC_train, "AUC_test": AUC_test
    })
res = pd.DataFrame(results).sort_values("AUC_test", ascending=False).reset_index(drop=True)
print(res)

# 모델적용 전체데이터 학습
model = models[res.loc[0, "Model"]]
y_pred = model.predict_proba(X_submission)[:, 1]
# print(y_pred)

# 제출파일 생성
pd.DataFrame({'pred': y_pred}).to_csv("result_01(2기).csv", index=False)

# 예측결과 확인
temp = pd.read_csv("result_01(2기).csv")
print(temp['pred'].describe())



              Model  AUC_train  AUC_test
0          AdaBoost   0.770532  0.708262
1  GradientBoosting   0.862112  0.702505
2       RandomForet   0.744530  0.667639
3          Logistic   0.613821  0.637775
4      DecisionTree   0.703950  0.634339
count    1221.000000
mean        0.412835
std         0.035228
min         0.273436
25%         0.389646
50%         0.412195
75%         0.434011
max         0.552332
Name: pred, dtype: float64
